# SUBQUERIES

A subquery is a query written inside another SQL query.
It is used to fetch intermediate results that are used by the outer query.

---

## Types

1. **Scalar Subquery**  
   Returns a single value (one row, one column).

2. **Row Subquery**  
   Returns multiple rows but only one column.

3. **Table Subquery (Derived Table)**  
   Returns multiple rows and columns and is usually used in the `FROM` clause.

4. **Correlated Subquery**  
   Depends on the outer query and runs once for each row.


# Independent Subquery - Scalar Subquery

# Practice questions


# question: Find the movie with highest profit

# method -1 (using subquery)
# Time Complexity: O(n)
SELECT *
FROM movies
WHERE profit = (SELECT MAX(profit) FROM movies);


# method -2 (without subquery - ORDER BY)
# Time Complexity: O(n log n)
SELECT *
FROM movies
ORDER BY profit DESC
LIMIT 1;


# question: Count movies whose rating is greater than average rating

# method -1 (using subquery)
# Time Complexity: O(n)
SELECT COUNT(*)
FROM movies
WHERE rating > (SELECT AVG(rating) FROM movies);


# method -2 (without subquery - JOIN)
# Time Complexity: O(n)
SELECT COUNT(*)
FROM movies m
JOIN (
    SELECT AVG(rating) AS avg_rating
    FROM movies
) t
ON m.rating > t.avg_rating;


# question: Find the highest rated movie of year 2000

# method -1 (using subquery)
# Time Complexity: O(n)
SELECT *
FROM movies
WHERE release_year = 2000
AND rating = (
    SELECT MAX(rating)
    FROM movies
    WHERE release_year = 2000
);


# method -2 (without subquery - ORDER BY)
# Time Complexity: O(n log n)
SELECT *
FROM movies
WHERE release_year = 2000
ORDER BY rating DESC
LIMIT 1;


# question: Find highest rated movie with votes greater than average votes

# method -1 (using subquery)
# Time Complexity: O(n)
SELECT *
FROM movies
WHERE votes > (SELECT AVG(votes) FROM movies)
AND rating = (
    SELECT MAX(rating)
    FROM movies
    WHERE votes > (SELECT AVG(votes) FROM movies)
);


# method -2 (without subquery - ORDER BY)
# Time Complexity: O(n log n)
SELECT *
FROM movies
WHERE votes > (SELECT AVG(votes) FROM movies)
ORDER BY rating DESC
LIMIT 1;


# Independent Subquery - Row Subquery(One Col Multi Rows)

# PRACTICE QUESTIONS

# question: Find all users who never ordered

# method -1 (using row subquery - NOT IN)
# Time Complexity: O(n + m)
SELECT *
FROM users
WHERE user_id NOT IN (
    SELECT user_id
    FROM orders
);


# method -2 (without subquery - LEFT JOIN)
# Time Complexity: O(n + m)
SELECT u.*
FROM users u
LEFT JOIN orders o
ON u.user_id = o.user_id
WHERE o.user_id IS NULL;


# question: Find all movies made by top 3 directors (by total gross)

# method -1 (using row subquery)
# Time Complexity: O(n log n)
SELECT *
FROM movies
WHERE director_id IN (
    SELECT director_id
    FROM movies
    GROUP BY director_id
    ORDER BY SUM(gross) DESC
    LIMIT 3
);


# method -2 (without subquery - JOIN with derived table)
# Time Complexity: O(n log n)
SELECT m.*
FROM movies m
JOIN (
    SELECT director_id
    FROM movies
    GROUP BY director_id
    ORDER BY SUM(gross) DESC
    LIMIT 3
) t
ON m.director_id = t.director_id;


# question: Find all movies of actors whose avg rating > 8.5 (votes >= 25000)

# method -1 (using row subquery)
# Time Complexity: O(n)
SELECT *
FROM campusx.movies_boxo
where star in (select star from movies
where votes>25000
group by star
having avg(score) > 8.5)


# method -2 (without subquery - JOIN + HAVING)
# Time Complexity: O(n)
SELECT DISTINCT m.*
FROM movies m
JOIN movie_cast mc ON m.movie_id = mc.movie_id
JOIN (
    SELECT mc.actor_id
    FROM movie_cast mc
    JOIN movies m2 ON mc.movie_id = m2.movie_id
    WHERE m2.votes >= 25000
    GROUP BY mc.actor_id
    HAVING AVG(m2.rating) > 8.5
) t
ON mc.actor_id = t.actor_id;


# Independent Subquery - Table Subquery(Multi Col Multi Row)

# question: Find the most profitable movie of each year

# method -1 (using row subquery)
# Time Complexity: O(n)
SELECT *
FROM movies m
WHERE profit = (
    SELECT MAX(profit)
    FROM movies
    WHERE release_year = m.release_year
);


# method -2 (without subquery - window function)
# Time Complexity: O(n)
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY release_year ORDER BY profit DESC) AS rnk
    FROM movies
) t
WHERE rnk = 1;



# question: Find highest rated movie of each genre (votes >= 25000)

# method -1 (using row subquery)
# Time Complexity: O(n)
SELECT *
FROM movies m
WHERE votes >= 25000
AND rating = (
    SELECT MAX(rating)
    FROM movies
    WHERE genre = m.genre
    AND votes >= 25000
);

# method -2 (without subquery - window function)
# Time Complexity: O(n)
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY genre ORDER BY rating DESC) AS rnk
    FROM movies
    WHERE votes >= 25000
) t
WHERE rnk = 1;


# question: Find movies of top 5 actor-director combos by total gross

# method -1 (using row subquery)
# Time Complexity: O(n log n)
SELECT m.*
FROM movies m
JOIN (
    SELECT star, director
    FROM movies
    GROUP BY star, director
    ORDER BY SUM(gross) DESC
    LIMIT 5
) t
ON m.star = t.star
AND m.director = t.director
WHERE m.gross = (
    SELECT MAX(gross)
    FROM movies
    WHERE star = m.star
    AND director = m.director
);


# Correlated Subquery
A correlated subquery is a subquery that depends on the outer query.


# question:
# Find all the movies that have a rating higher than the average rating of movies in the same genre

# method -1 (with subquery : CORRELATED SUBQUERY)
# Time Complexity: O(n^2)

SELECT *
FROM movies m1
WHERE rating > (
    SELECT AVG(rating)
    FROM movies m2
    WHERE m2.genre = m1.genre
);

# method -2 (without subquery : JOIN)
# Time Complexity: O(n)

SELECT m.*
FROM movies m
JOIN (
    SELECT genre, AVG(rating) AS avg_rating
    FROM movies
    GROUP BY genre
) g
ON m.genre = g.genre
WHERE m.rating > g.avg_rating;


# question:
# Find the favorite food of each customer

# method -1 (with subquery : CORRELATED SUBQUERY)
# Time Complexity: O(n^2)

SELECT DISTINCT o1.customer_id, o1.food_item
FROM orders o1
WHERE o1.food_item = (
    SELECT o2.food_item
    FROM orders o2
    WHERE o2.customer_id = o1.customer_id
    GROUP BY o2.food_item
    ORDER BY COUNT(*) DESC
    LIMIT 1
);



# Using with select

# question:
# Get the percentage of votes for each movie compared to total votes

# method -1 (with subquery in SELECT : SCALAR SUBQUERY)
# Time Complexity: O(n)

SELECT
    title,
    (votes * 100.0) / (SELECT SUM(votes) FROM movies) AS vote_percentage
FROM movies;


# question:
# Display movie name, genre, score and average score of that genre

# method -1 (with subquery in SELECT : CORRELATED SUBQUERY)
# Time Complexity: O(n^2)

SELECT
    title,
    genre,
    score,
    (SELECT AVG(score)
     FROM movies
     WHERE genre = m.genre) AS genre_avg_score
FROM movies m;


# Using with From

# question:
# Display movie name, genre, score and average score of that genre


# method -1 (with subquery in FROM : DERIVED TABLE)
# Time Complexity: O(n)

SELECT
    m.title,
    m.genre,
    m.score,
    g.avg_score
FROM movies m
JOIN (
    SELECT genre, AVG(score) AS avg_score
    FROM movies
    GROUP BY genre
) g
ON m.genre = g.genre;


# Using with having

# question:
# Find genres having avg score > avg score of all the movies

# method -1 (with subquery in HAVING)
# Time Complexity: O(n)

SELECT genre
FROM movies
GROUP BY genre
HAVING AVG(score) > (
    SELECT AVG(score)
    FROM movies
);


# Using with insert

# Populate loyal_users table with only those users who have ordered food more than 3 times.

CREATE TABLE loyal_users (
    user_id INT PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    money INT DEFAULT 0
);


INSERT INTO casestudy.loyal_users (user_id, name)
SELECT 
    o.user_id,
    u.name
FROM casestudy.zomato_orders o
JOIN casestudy.zomato_users u
    ON o.user_id = u.user_id
GROUP BY 
    o.user_id,
    u.name
HAVING COUNT(*) > 3;


# Using with update

# question:
# Populate money column of loyal_users using orders table
# Provide 10% app money based on total order value

# method  (with subquery in UPDATE : CORRELATED SUBQUERY)
# Time Complexity: O(n^2)

UPDATE casestudy.loyal_users lu
SET money = (
    SELECT SUM(o.amount) * 0.1
    FROM casestudy.zomato_orders o
    WHERE o.user_id = lu.user_id
);


# Using with delete

# question:
# Delete all the customers who have never ordered

# method -1 (with subquery : NOT IN)
# Time Complexity: O(n + m)

DELETE
FROM casestudy.zomato_users
WHERE user_id NOT IN (
    SELECT DISTINCT user_id
    FROM casestudy.zomato_orders
);
